# Orchid ResNet-18 E2 ? HSV colour branch
Select a GPU runtime and run all cells. This compares a fresh HSV run with the winning E1 baseline. It keeps plain cross-entropy, the E1 split, seed, learning rate and training budget. A 16-bin histogram for each of H, S and V is computed on a subsampled grid of the same crop seen by ResNet. Only CF and CL receive the extra 48 inputs. Panicle is tracked as a secondary metric. The test split is excluded.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
import pandas as pd
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime.'
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
dataset_zip = drive_root / 'Colab Notebooks/orchid-project/Orchid Flowers Dataset-v1.1.zip'
if not dataset_zip.is_file():
    raise FileNotFoundError(f'Update dataset_zip to your existing data archive: {dataset_zip}')
repo = Path('/content/orchid-e2')
revision = 'ba3ca576cedc4215a87e7790a6d83687716e8e42'
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'a-data',
                    'https://github.com/satyamchauhan49/orchid-identification.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', revision], check=True)
os.chdir(repo)
manifest = repo / 'outputs/split_manifest_grouped.csv'
frame = pd.read_csv(manifest, dtype={'image_id': str})
assert frame['split'].value_counts().to_dict() == {'train': 4579, 'calib': 509, 'val': 1273, 'test': 795}
assert frame['image_id'].is_unique and frame.groupby('phash_cluster')['split'].nunique().max() == 1
raw = Path('/content/orchid-e1-dataset')
if not list(raw.rglob('Orchid_Images')):
    with zipfile.ZipFile(dataset_zip) as archive:
        archive.extractall(raw)
dataset_root, = [path.parent for path in raw.rglob('Orchid_Images') if path.is_dir()]
assert all((dataset_root / path).is_file() for path in frame['path'])
reference = repo / 'outputs/orchid-e1/e36dd98/e36dd98'
import json
baseline = json.loads((reference / 'resnet18_base/selection.json').read_text())
assert baseline['learning_rate'] == 0.0001 and baseline['rows'] == 1273
save_root = drive_root / 'orchid-e2' / 'e36dd98'
save_root.mkdir(parents=True, exist_ok=True)
print('E1 reference:', baseline['run'])
print('E2 output:', save_root)

In [ ]:
name = 'resnet18_e2_hsv_cf_cl'
destination = save_root / name
if (destination / '.complete').is_file():
    print('Reusing completed run:', destination)
else:
    trainer = repo / 'models_resnet/train_resnet.py'
    variant = trainer.with_name('train_resnet_e2_colab.py')
    source = trainer.read_text(encoding='utf-8')
    hsv_module = repo / 'models_resnet/model_hsv_colab.py'
    hsv_module.write_text('''import torch
from torch import nn

from model import OrchidMultiTaskResNet18


class OrchidHSVResNet18(OrchidMultiTaskResNet18):
    def __init__(self, num_species, feature_classes, pretrained, mean, std):
        super().__init__(num_species, feature_classes, pretrained)
        self.register_buffer('rgb_mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('rgb_std', torch.tensor(std).view(1, 3, 1, 1))
        for head in ('CF', 'CL'):
            self.feature_heads[head] = nn.Linear(self.embedding_dim + 48, feature_classes[head])

    def hsv_histogram(self, x):
        rgb = (x[:, :, ::4, ::4].float() * self.rgb_std + self.rgb_mean).clamp(0, 1)
        red, green, blue = rgb.unbind(dim=1)
        value, channel = rgb.max(dim=1)
        chroma = value - rgb.min(dim=1).values
        safe_chroma = chroma.clamp_min(1e-8)
        hue = torch.where(channel == 0, (green - blue) / safe_chroma,
                          torch.where(channel == 1, (blue - red) / safe_chroma + 2,
                                      (red - green) / safe_chroma + 4))
        hue = (hue / 6).remainder(1)
        saturation = chroma / value.clamp_min(1e-8)
        bins = (torch.stack((hue, saturation, value), dim=1).flatten(2) * 16).long().clamp(0, 15)
        histograms = rgb.new_zeros((x.shape[0], 3, 16))
        histograms.scatter_add_(2, bins, rgb.new_ones(bins.shape))
        return (histograms / bins.shape[-1]).flatten(1)

    def forward(self, x):
        features = self.backbone(x).flatten(1)
        colour_features = torch.cat((features, self.hsv_histogram(x).to(features.dtype)), dim=1)
        predictions = {'species': self.species_head(features)}
        predictions.update({head: layer(colour_features if head in ('CF', 'CL') else features)
                            for head, layer in self.feature_heads.items()})
        return predictions
''', encoding='utf-8')
    changes = [
        ('from model import OrchidMultiTaskResNet18',
         'from model_hsv_colab import OrchidHSVResNet18'),
        ('OrchidMultiTaskResNet18(n_species, feature_classes, args.pretrained)',
         'OrchidHSVResNet18(n_species, feature_classes, args.pretrained, mean, std)'),
        ("['train', 'calib', 'val', 'test']", "['train', 'calib', 'val']"),
        ("['calib', 'val', 'test']", "['calib', 'val']"),
    ]
    for before, after in changes:
        if source.count(before) != 1:
            raise RuntimeError(f'Unexpected trainer source: {before}')
        source = source.replace(before, after)
    variant.write_text(source, encoding='utf-8')
    subprocess.run([sys.executable, str(variant),
                    '--dataset-root', str(dataset_root), '--manifest', str(manifest),
                    '--run-name', name, '--lr', str(baseline['learning_rate']),
                    '--epochs', '15', '--batch-size', '32', '--workers', '2',
                    '--seed', '42', '--patience', '4', '--pretrained'], check=True)
    shutil.copytree(repo / 'outputs' / name, destination, dirs_exist_ok=True)
    (destination / '.complete').touch()

In [ ]:
rates = {name: baseline['learning_rate']}
import json
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score

val = frame.loc[frame['split'] == 'val'].set_index('image_id')
shared_maps = json.loads((repo / 'outputs/label_maps.json').read_text())
heads = ('species', 'T', 'NF', 'In', 'LC', 'CF', 'CL')
summary, details = [], []
for name, rate in rates.items():
    run = save_root / name
    history = pd.read_csv(run / 'training_history.csv')
    best = history.loc[history['val_loss'].idxmin()]
    maps = json.loads((run / 'class_maps.json').read_text())
    with np.load(run / 'val.npz', allow_pickle=False) as data:
        ids = pd.Index(data['ids'].astype(str))
        assert ids.is_unique and set(ids) == set(val.index)
        for head in heads:
            labels = sorted(maps[head], key=maps[head].get)
            assert labels == [str(label) for label in shared_maps[head]], head
            y = data[f'targets_{head}']
            assert np.array_equal(y, val.loc[ids, head].astype(str).map(maps[head]).to_numpy())
            key = 'species_logits' if head == 'species' else f'logits_{head}'
            logits = data[key]
            assert logits.shape == (len(y), len(labels)) and np.isfinite(logits).all()
            ranked = np.argsort(-logits, axis=1, kind='stable')
            predicted = ranked[:, 0]
            metrics = dict(run=name, learning_rate=float(rate), head=head, rows=len(y),
                           best_epoch=int(best['epoch']), val_loss=float(best['val_loss']),
                           top1=(predicted == y).mean(), macro_f1=f1_score(y, predicted, average='macro'))
            if head == 'species':
                for k in (3, 5):
                    metrics[f'top{k}'] = (ranked[:, :k] == y[:, None]).any(axis=1).mean()
            else:
                cm = confusion_matrix(y, predicted, labels=range(len(labels)))
                pd.DataFrame(cm, index=labels, columns=labels).rename_axis('actual').to_csv(
                    run / f'val_confusion_{head}.csv')
                for i, label in enumerate(labels):
                    tp, positives = int(cm[i, i]), int(cm[i, :].sum())
                    fp, negatives = int(cm[:, i].sum()) - tp, len(y) - positives
                    details.append(dict(run=name, head=head, label=label, support=positives,
                                        tpr=tp / positives if positives else np.nan,
                                        tnr=(negatives - fp) / negatives if negatives else np.nan))
            summary.append(metrics)
summary, details = pd.DataFrame(summary), pd.DataFrame(details)
base_summary = pd.read_csv(reference / 'R1_validation_summary.csv')
base_details = pd.read_csv(reference / 'R1_validation_by_class.csv')
base_summary = base_summary.loc[base_summary['run'] == baseline['run']].drop(columns='run')
base_details = base_details.loc[base_details['run'] == baseline['run']].drop(columns='run')
heads_r2 = summary.merge(base_summary, on='head', suffixes=('_e2', '_e1'), validate='one_to_one')
classes_r2 = details.merge(base_details, on=['head', 'label'], suffixes=('_e2', '_e1'),
                           validate='one_to_one')
assert len(heads_r2) == len(summary) and len(classes_r2) == len(details)
assert (heads_r2['rows_e2'] == heads_r2['rows_e1']).all()
assert (classes_r2['support_e2'] == classes_r2['support_e1']).all()
heads_r2['colour_branch'] = 'HSV_16_bins_per_channel'
classes_r2['colour_branch'] = 'HSV_16_bins_per_channel'
for metric in ('top1', 'top3', 'top5', 'macro_f1', 'val_loss'):
    heads_r2[f'{metric}_delta'] = heads_r2[f'{metric}_e2'] - heads_r2[f'{metric}_e1']
for metric in ('tpr', 'tnr'):
    classes_r2[f'{metric}_delta'] = classes_r2[f'{metric}_e2'] - classes_r2[f'{metric}_e1']
heads_r2.to_csv(save_root / 'R2_hsv_summary.csv', index=False)
classes_r2.to_csv(save_root / 'R2_hsv_by_class.csv', index=False)
display(heads_r2[['head', 'top1_e1', 'top1_e2', 'top1_delta', 'macro_f1_delta']])
display(classes_r2.loc[classes_r2['label'].isin(['GreenYellow', 'Panicle', 'WhiteYellow']),
                       ['head', 'label', 'support_e2', 'tpr_e1', 'tpr_e2', 'tpr_delta', 'tnr_delta']])
print('R2 results:', save_root)